# CM4044 MP2-30 · How Noisy Is Toxicity? Label Noise and Ensemble Robustness on Tox21

**Question.** Toxicity labels come from real assays with real error rates. How much label noise can training pipelines tolerate, and which defences actually work?

This notebook is **self-contained**: every line of code used in the project is in the cells below. Run it top to bottom on Google Colab (*Runtime → Change runtime type → T4 GPU*).

| Section | Content | Time on a Colab T4 |
|---|---|---|
| 0 | Setup and configuration | ~2 min |
| 1 | Data curation (curated dataset + curation log) | ~1 min |
| 2 | **Task 1**: natural noise in Tox21 | ~1 min |
| 3–5 | Noise injection, models + training, noise detectors (definitions) | – |
| 6 | **Tasks 2 + 3**: noise level × defence × model × seed grid | ~4–5 h in total (resumable) |
| 7 | Results tables and figures (baseline-vs-main comparison table) | 1 min |
| 8 | **Task 4**: do detectors flag *real* suspect labels? | ~10 min |
| 9 | **Stretch**: multi-task noise interaction | ~30 min |
| 10 | Model cards and the submission zip | 1 min |

**Baseline vs. main models.** The baseline is the *undefended* model (standard training). Each defence is a main model, trained with identical splits, seeds, hyperparameters and metrics:
* label smoothing (`ls`)
* forward loss correction with an estimated noise matrix (`fc`), and with the true one as an upper bound (`fc_oracle`)
* confident learning (`cl`)
* ensemble-disagreement filtering (`ed`)

The defences are applied to a multi-task MLP on ECFP4, a GINE graph neural network and (for the cleaning defences) a random-forest classical baseline.

**Colab disconnects are not a problem.** All outputs go to Google Drive (`MyDrive/CM4044_MP2_results`), and every training run is saved as soon as it finishes. After a disconnect, re-run sections 0–5 (fast) and then the section you were in: finished runs are skipped.

**Splitting the work across the group.** In section 6, each member can run a different seed (`seeds=[0]`, `[1]`, `[2]`) on their own Colab account. Afterwards, copy all `results/runs/*.json` and `results/detection/*.json` files into one Drive folder and run section 7.

## 0 · Setup and configuration
Installs the three packages Colab does not ship with (RDKit, PyTorch Geometric, cleanlab). It then mounts Google Drive and makes the Drive folder the working directory.

In [ ]:
import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit==2026.9.1",
                    "torch_geometric==2.8.1", "cleanlab==2.9.0", "tabulate"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/CM4044_MP2_results"
else:                                    # running locally
    WORKDIR = os.path.abspath("mp2_work")
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print("working directory:", os.getcwd())

In [ ]:
import hashlib, itertools, json, platform, random, time, urllib.request, warnings
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Image, Markdown, display
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
from scipy.stats import fisher_exact
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import KFold
from cleanlab.count import compute_confident_joint, estimate_latent
from cleanlab.filter import find_label_issues

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore", category=UserWarning)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# ---------------- Configuration: every setting used in the project ----------------
QUICK = False   # True = fast smoke test (1 seed, 1 noise level, few epochs) written to a separate folder

RAW_URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz"   # MoleculeNet release
RAW_SHA256 = "45d09792492ce049039dd24aa27b07fc79ce20c573187d4d90bcd178c0c0d360"
DATA_DIR = Path("data")
RAW_PATH = DATA_DIR / "raw" / "tox21.csv.gz"
CURATED_PATH = DATA_DIR / "tox21_curated.csv"
CURATION_LOG = DATA_DIR / "curation_log.md"
CONFLICTS_PATH = DATA_DIR / "duplicate_conflicts.csv"
CACHE_DIR = DATA_DIR / "cache"
RESULTS_DIR = Path("results_quick" if QUICK else "results")
FIG_DIR = Path("figures_quick" if QUICK else "figures")
MODELS_DIR = Path("models")
for d in (RAW_PATH.parent, CACHE_DIR, RESULTS_DIR / "cache", FIG_DIR, MODELS_DIR / "checkpoints", MODELS_DIR / "cards"):
    d.mkdir(parents=True, exist_ok=True)

TASKS = ["NR-AR", "NR-AR-LBD", "NR-AhR", "NR-Aromatase", "NR-ER", "NR-ER-LBD",
         "NR-PPAR-gamma", "SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53"]
# Assays probing the same receptor (full-length vs ligand-binding domain). A molecule active in one
# and inactive in the other has a "suspect" label: at least one readout is probably wrong.
RELATED_PAIRS = [("NR-AR", "NR-AR-LBD"), ("NR-ER", "NR-ER-LBD")]
SPLIT_FRACS = (0.8, 0.1, 0.1)            # train / valid / test, Bemis-Murcko scaffold split
ECFP_RADIUS, ECFP_BITS = 2, 2048         # ECFP4

NOISE_KINDS = ["sym", "cc"]              # symmetric, class-conditional
NOISE_RATES = [0.05, 0.10, 0.20]
SEEDS = [0, 1, 2]

# identical hyperparameters for every defence -> fair comparison
MLP_HP = dict(hidden=[1024, 256], dropout=0.3, lr=1e-3, weight_decay=1e-5, batch_size=128, max_epochs=80, patience=12)
GNN_HP = dict(hidden=128, layers=4, dropout=0.2, lr=1e-3, weight_decay=1e-5, batch_size=128, max_epochs=80, patience=12)
RF_HP = dict(n_estimators=100, min_samples_leaf=1, max_features="sqrt", n_jobs=-1)

DEFENCES = ["none", "ls", "fc", "fc_oracle", "cl", "ed"]
LABEL_SMOOTHING_EPS = 0.1
OOF_FOLDS = 5                            # cross-validation folds for out-of-fold detector probabilities
DETECTOR_MEMBERS = ["lr", "knn", "mlp"]  # three deliberately different learners
DETECTOR_MLP_EPOCHS = 25
ED_MIN_VOTES = 2                         # ensemble filter: majority of the 3 members
SAVE_CONDITIONS = ["clean", "sym_0.20", "cc_0.20"]   # seed-0 checkpoints kept for the submission (fp16)

if QUICK:
    SEEDS, NOISE_KINDS, NOISE_RATES, OOF_FOLDS = [0], ["sym"], [0.10], 3
    MLP_HP.update(max_epochs=5, patience=3); GNN_HP.update(max_epochs=2, patience=2)
    DETECTOR_MLP_EPOCHS = 3
    SAVE_CONDITIONS = ["clean", "sym_0.10"]

## 1 · Data curation
Raw Tox21 (7,831 records × 12 assays) is processed in four steps:
1. **Standardise.** RDKit cleanup, keep the largest organic fragment (removes salts), neutralise charges, canonical SMILES.
2. **Drop invalid records.** Unparseable, inorganic (no carbon) and tiny (<3 heavy atoms) structures are removed.
3. **Merge duplicate parent structures.** Agreeing labels are kept, conflicts are resolved by strict majority, and ties become missing. Every conflict is logged, because replicate disagreement *is* natural label noise.
4. **Scaffold split.** A deterministic Bemis–Murcko scaffold split (80/10/10).

Every decision and its record count is written to `data/curation_log.md`.

In [ ]:
_cleanup = rdMolStandardize.CleanupParameters()
_largest = rdMolStandardize.LargestFragmentChooser(preferOrganic=True)
_uncharger = rdMolStandardize.Uncharger()


def download_raw():
    if not RAW_PATH.exists():
        urllib.request.urlretrieve(RAW_URL, RAW_PATH)
    digest = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
    assert digest == RAW_SHA256, f"raw file checksum mismatch: {digest}"
    return digest


def standardise(smiles):
    """Return (standardised SMILES, status); status != 'ok' means the record is dropped."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:   # mostly hypervalent metal salts: strip the counter-ion without sanitising first
        mol = Chem.MolFromSmiles(smiles, sanitize=False)
        try:
            mol = _largest.choose(mol)
            Chem.SanitizeMol(mol)
        except Exception:
            return None, "unparseable"
    try:
        mol = rdMolStandardize.Cleanup(mol, _cleanup)   # normalise groups, disconnect metals
        mol = _largest.choose(mol)                       # parent = largest organic fragment
        mol = _uncharger.uncharge(mol)                   # neutralise
        Chem.SanitizeMol(mol)
    except Exception:
        return None, "standardisation_failed"
    if not any(a.GetAtomicNum() == 6 for a in mol.GetAtoms()):
        return None, "no_carbon"
    if mol.GetNumHeavyAtoms() < 3:
        return None, "too_small"
    return Chem.MolToSmiles(mol), "ok"


def scaffold(smiles):
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(smiles=smiles, includeChirality=False)
    except Exception:
        return ""


def scaffold_split(smiles_list, fracs=SPLIT_FRACS):
    """Deterministic scaffold split: largest scaffold groups fill train, rarer chemotypes go to valid/test."""
    groups = defaultdict(list)
    for i, s in enumerate(smiles_list):
        groups[scaffold(s)].append(i)
    ordered = sorted(groups.values(), key=lambda g: (len(g), g[0]), reverse=True)
    split, filled, n = np.empty(len(smiles_list), dtype=object), 0, len(smiles_list)
    for g in ordered:
        split[g] = ("train" if filled + len(g) <= fracs[0] * n else
                    "valid" if filled + len(g) <= (fracs[0] + fracs[1]) * n else "test")
        filled += len(g)
    return split, len(groups)


def merge_duplicates(df):
    """Merge records with the same standardised structure; log every label conflict."""
    rows, conflicts, replicated = [], [], 0
    for smi, g in df.groupby("smiles", sort=False):
        rec = {"smiles": smi, "mol_id": ";".join(g["mol_id"]), "smiles_raw": g["smiles_raw"].iloc[0],
               "n_records": len(g)}
        for t in TASKS:
            vals = g[t].dropna().values
            replicated += len(vals) >= 2
            if len(vals) == 0:
                rec[t] = np.nan
            elif len(set(vals)) == 1:
                rec[t] = vals[0]
            else:
                n_pos, n_neg = int((vals == 1).sum()), int((vals == 0).sum())
                rec[t] = 1.0 if n_pos > n_neg else 0.0 if n_neg > n_pos else np.nan
                conflicts.append(dict(smiles=smi, mol_id=rec["mol_id"], task=t, n_active=n_pos,
                                      n_inactive=n_neg, resolved=rec[t]))
        rows.append(rec)
    return pd.DataFrame(rows), pd.DataFrame(conflicts), replicated


def curate_dataset():
    digest = download_raw()
    raw = pd.read_csv(RAW_PATH).rename(columns={"smiles": "smiles_raw"})
    n_raw = len(raw)
    out = raw["smiles_raw"].apply(standardise)
    raw["smiles"], raw["status"] = [o[0] for o in out], [o[1] for o in out]
    ok = raw[raw["status"] == "ok"].copy()
    dropped = raw.loc[raw["status"] != "ok", "status"].value_counts().to_dict()
    n_changed = int((ok["smiles"] != ok["smiles_raw"]).sum())
    n_multi = int(raw["smiles_raw"].str.contains(r"\.").sum())
    no_label = ok[TASKS].isna().all(axis=1)
    ok = ok[~no_label]
    merged, conflicts, replicated = merge_duplicates(ok)
    n_ties = int(conflicts["resolved"].isna().sum())
    merged["split"], n_scaffolds = scaffold_split(merged["smiles"].tolist())

    log = ["# Curation log - Tox21\n", f"Source: `{RAW_URL}`  ", f"SHA-256: `{digest}`\n",
           "| Step | Decision | Records after |", "|---|---|---|",
           f"| 0 | Raw file: {n_raw} records x {len(TASKS)} assays | {n_raw} |",
           f"| 1 | RDKit standardisation: cleanup/normalise, largest organic fragment ({n_multi} multi-component "
           f"records = salts/mixtures), neutralise, canonical SMILES; {n_changed} SMILES changed. Tautomers not "
           f"canonicalised; counter-ions assumed not to drive activity | {n_raw} |",
           f"| 2 | Dropped unparseable / inorganic / tiny records: {dropped} | {n_raw - sum(dropped.values())} |",
           f"| 3 | Dropped records with no label: {int(no_label.sum())} | {len(ok)} |",
           f"| 4 | Merged duplicate parents ({int((merged.n_records > 1).sum())} structures had >1 record). "
           f"Of {replicated} (structure, assay) cells measured more than once, {len(conflicts)} "
           f"({len(conflicts) / replicated:.1%}) conflict (`duplicate_conflicts.csv`): "
           f"{len(conflicts) - n_ties} majority-resolved, {n_ties} ties set to missing | {len(merged)} |",
           f"| 5 | Scaffold split 80/10/10: {merged['split'].value_counts().to_dict()}, {n_scaffolds} scaffolds | {len(merged)} |",
           f"| 6 | No subsampling needed ({len(merged)} <= 10k budget) | {len(merged)} |",
           "\n| Assay | labelled | actives | active rate |", "|---|---|---|---|"]
    for t in TASKS:
        y = merged[t]
        log.append(f"| {t} | {int(y.notna().sum())} | {int((y == 1).sum())} | {(y == 1).sum() / y.notna().sum():.3f} |")
    log.append("\nTest-set labels are never modified by any noise-injection or cleaning step.")
    merged[["mol_id", "smiles_raw", "smiles", "n_records"] + TASKS + ["split"]].to_csv(CURATED_PATH, index=False)
    conflicts.to_csv(CONFLICTS_PATH, index=False)
    CURATION_LOG.write_text("\n".join(log) + "\n")


if not CURATED_PATH.exists():
    curate_dataset()
display(Markdown(CURATION_LOG.read_text()))

In [ ]:
class Dataset:
    """Curated Tox21: label matrix Y (n x 12, NaN = missing) + split row indices."""

    def __init__(self):
        self.df = pd.read_csv(CURATED_PATH)
        self.smiles = self.df["smiles"].tolist()
        self.Y = self.df[TASKS].to_numpy(dtype=np.float32)
        split = self.df["split"].to_numpy()
        self.idx = {s: np.where(split == s)[0] for s in ("train", "valid", "test")}
        self.idx["dev"] = np.concatenate([self.idx["train"], self.idx["valid"]])  # rows whose labels may be noisy


def ecfp4(smiles):
    path = CACHE_DIR / f"ecfp4_{ECFP_BITS}_{len(smiles)}.npy"
    if path.exists():
        return np.load(path)
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=ECFP_RADIUS, fpSize=ECFP_BITS)
    X = np.zeros((len(smiles), ECFP_BITS), dtype=np.uint8)
    for i, s in enumerate(smiles):
        DataStructs.ConvertToNumpyArray(gen.GetFingerprint(Chem.MolFromSmiles(s)), X[i])
    np.save(path, X)
    return X


def build_graphs(smiles):
    """PyG graphs with OGB-style categorical atom/bond features; `row` = index into the label matrix."""
    from torch_geometric.utils import from_smiles
    path = CACHE_DIR / f"graphs_{len(smiles)}.pt"
    if path.exists():
        return torch.load(path, weights_only=False)
    gs = []
    for i, s in enumerate(smiles):
        g = from_smiles(s)
        g.row = torch.tensor([i])
        del g.smiles
        gs.append(g)
    torch.save(gs, path)
    return gs


ds = Dataset()
X = ecfp4(ds.smiles)
GRAPHS = build_graphs(ds.smiles)
print(f"{len(ds.smiles)} molecules | ECFP4 {X.shape} | {len(GRAPHS)} graphs | split sizes:",
      {k: len(v) for k, v in ds.idx.items()})

## 2 · Task 1: natural noise in Tox21
Three views of the noise that is *already* in the labels:
1. **Class imbalance** per assay.
2. **Replicate disagreement.** The same parent structure was tested more than once and gave different answers.
3. **Cross-assay disagreement** between assays that probe the same receptor:
   * NR-AR (full-length, MDA cell line) vs. NR-AR-LBD (ligand-binding domain)
   * NR-ER (full-length, BG1 cell line, luciferase) vs. NR-ER-LBD

   Where the two readouts of a pair disagree, both labels are marked **suspect** (used in Task 4).

In [ ]:
def cohen_kappa(a, b):
    po = np.mean(a == b)
    pe = np.mean(a) * np.mean(b) + (1 - np.mean(a)) * (1 - np.mean(b))
    return (po - pe) / (1 - pe)


def suspect_matrix(Y):
    """suspect[i, t]: both related-assay labels observed and they disagree.
    paired[i, t]: both related-assay labels observed (the comparison population)."""
    suspect, paired = np.zeros_like(Y, dtype=bool), np.zeros_like(Y, dtype=bool)
    for a, b in RELATED_PAIRS:
        i, j = TASKS.index(a), TASKS.index(b)
        both = ~np.isnan(Y[:, i]) & ~np.isnan(Y[:, j])
        for k in (i, j):
            paired[:, k] |= both
            suspect[:, k] |= both & (Y[:, i] != Y[:, j])
    return suspect, paired


def natural_noise_analysis(Y):
    # (1) class imbalance
    imb = pd.DataFrame([dict(assay=t, labelled=int((~np.isnan(Y[:, k])).sum()), actives=int(np.nansum(Y[:, k])))
                        for k, t in enumerate(TASKS)])
    imb["active_rate"] = imb.actives / imb.labelled
    imb["imbalance_ratio"] = (imb.labelled - imb.actives) / imb.actives
    imb.to_csv(RESULTS_DIR / "natural_imbalance.csv", index=False)

    # (2) replicate disagreement, from the raw file
    raw = pd.read_csv(RAW_PATH)
    raw["parent"] = [standardise(s)[0] for s in raw["smiles"]]
    dup = raw.dropna(subset=["parent"])
    dup = dup[dup.duplicated("parent", keep=False)]
    rep = []
    for t in TASKS:
        g = dup.groupby("parent")[t]
        measured = g.count() >= 2
        conflict = g.agg(lambda v: v.dropna().nunique() > 1)[measured]
        rep.append(dict(assay=t, replicated=int(measured.sum()), conflicting=int(conflict.sum()),
                        conflict_rate=float(conflict.mean())))
    rep = pd.DataFrame(rep)
    rep.to_csv(RESULTS_DIR / "natural_replicates.csv", index=False)

    # (3) agreement between every pair of assays
    pairs = []
    for (i, a), (j, b) in itertools.combinations(enumerate(TASKS), 2):
        both = ~np.isnan(Y[:, i]) & ~np.isnan(Y[:, j])
        ya, yb = Y[both, i], Y[both, j]
        pairs.append(dict(assay_a=a, assay_b=b, n_both=int(both.sum()),
                          both_active=int(((ya == 1) & (yb == 1)).sum()),
                          a_only=int(((ya == 1) & (yb == 0)).sum()), b_only=int(((ya == 0) & (yb == 1)).sum()),
                          kappa=cohen_kappa(ya, yb), related=(a, b) in RELATED_PAIRS))
    pairs = pd.DataFrame(pairs)
    pairs.to_csv(RESULTS_DIR / "natural_pairs.csv", index=False)
    rel = pairs[pairs.related].copy()
    rel["disagree_rate_among_actives"] = (rel.a_only + rel.b_only) / (rel.a_only + rel.b_only + rel.both_active)
    rel.to_csv(RESULTS_DIR / "natural_related_pairs.csv", index=False)
    suspect, paired = suspect_matrix(Y)

    display(Markdown("**Class imbalance**"), imb.round(3))
    display(Markdown(f"**Replicate disagreement** - overall "
                     f"{rep.conflicting.sum() / rep.replicated.sum():.1%} of replicated measurements conflict"),
            rep.round(3))
    display(Markdown(f"**Related-assay disagreement** - {suspect.sum()} suspect labels out of "
                     f"{paired.sum()} paired labels"), rel.round(3))

    fig, ax = plt.subplots(figsize=(7, 3.6))
    ax.bar(imb.assay, imb.active_rate * 100, color="#3b6ea8")
    for k, v in enumerate(imb.imbalance_ratio):
        ax.text(k, imb.active_rate[k] * 100 + 0.3, f"1:{v:.0f}", ha="center", fontsize=7)
    ax.set_ylabel("actives (% of labelled)")
    ax.set_title("Class imbalance per assay (inactive:active ratio)")
    ax.tick_params(axis="x", rotation=60)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_imbalance.png", dpi=200); plt.show()

    K = np.eye(len(TASKS))
    for _, r in pairs.iterrows():
        i, j = TASKS.index(r.assay_a), TASKS.index(r.assay_b)
        K[i, j] = K[j, i] = r.kappa
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(K, cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(len(TASKS)), TASKS, rotation=70, fontsize=7)
    ax.set_yticks(range(len(TASKS)), TASKS, fontsize=7)
    for i in range(len(TASKS)):
        for j in range(len(TASKS)):
            if i != j:
                ax.text(j, i, f"{K[i, j]:.2f}", ha="center", va="center", fontsize=5,
                        color="white" if K[i, j] < 0.5 else "black")
    fig.colorbar(im, ax=ax, label="Cohen's kappa")
    ax.set_title("Cross-assay label agreement")
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_assay_kappa.png", dpi=200); plt.show()


natural_noise_analysis(ds.Y)

## 3 · Task 2: controlled synthetic label noise
Noise is injected per assay into the **train + valid** labels only. The test set keeps its original labels, so test scores measure true performance. The validation set used for early stopping is noisy too, because a real modeller never has clean labels to tune on.

* **`sym` (symmetric).** Exactly *p* × (number of labelled molecules) labels are flipped, chosen at random regardless of class. With only ~5 % actives, this mostly creates *false actives*.
* **`cc` (class-conditional, prior-preserving).** A fraction *p* of the actives become inactive (missed actives: cytotoxicity masking, solubility). The *same number* of inactives become active (artefacts: autofluorescence, luciferase interference). The class balance looks unchanged, so this noise is invisible in the label statistics.

In [ ]:
def inject_noise(Y, rows, kind, rate, seed):
    """Return (noisy label matrix, boolean mask of flipped labels)."""
    Yn, flipped = Y.copy(), np.zeros_like(Y, dtype=bool)
    if kind == "none" or rate == 0:
        return Yn, flipped
    rng = np.random.default_rng(10_000 + seed)
    for t in range(Y.shape[1]):
        obs = rows[~np.isnan(Y[rows, t])]
        if kind == "sym":
            pick = rng.choice(obs, size=int(round(rate * len(obs))), replace=False)
        else:  # "cc"
            pos, neg = obs[Y[obs, t] == 1], obs[Y[obs, t] == 0]
            k = int(round(rate * len(pos)))
            pick = np.concatenate([rng.choice(pos, k, replace=False), rng.choice(neg, k, replace=False)])
        Yn[pick, t] = 1 - Y[pick, t]
        flipped[pick, t] = True
    return Yn, flipped


def condition_name(kind, rate):
    return "clean" if kind == "none" or rate == 0 else f"{kind}_{rate:.2f}"


for kind in ("sym", "cc"):
    Yn, fl = inject_noise(ds.Y, ds.idx["dev"], kind, 0.10, seed=0)
    dev = ds.idx["dev"]
    print(f"{kind} 10%: flipped {fl.sum()} labels; actives in train+valid {int(np.nansum(ds.Y[dev]))} -> "
          f"{int(np.nansum(Yn[dev]))}")

## 4 · Models and training
* **MLP**: multi-task, on 2048-bit ECFP4, layers 1024 → 256 with BatchNorm, ReLU and dropout; 12 sigmoid outputs (~2.4 M parameters).
* **GIN**: a GINE graph network (Hu et al. 2020) in PyTorch Geometric. Atom and bond features are categorical embeddings; 4 residual message-passing layers of width 128; mean+max pooling; 12 sigmoid outputs (~0.34 M parameters).
* **RF**: classical baseline, one random forest per assay on ECFP4.

All neural models use a **masked** binary cross-entropy: missing labels contribute nothing, including labels removed by a cleaning defence. Two defences act inside the loss:
* **Label smoothing**: targets become *y*(1−ε) + ε/2.
* **Forward loss correction** (Patrini et al. 2017): the predicted clean probability *p* is passed through the per-assay noise channel, *q* = *p*(1−ρ₁₀) + (1−*p*)ρ₀₁, and the BCE is taken against *q*.

Training uses Adam with early stopping on mean validation ROC-AUC.

In [ ]:
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_max_pool, global_mean_pool
from torch_geometric.utils.smiles import e_map, x_map


class MLP(nn.Module):
    def __init__(self, n_in, n_out, hidden=(1024, 256), dropout=0.3):
        super().__init__()
        layers, d = [], n_in
        for h in hidden:
            layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        self.net = nn.Sequential(*layers, nn.Linear(d, n_out))

    def forward(self, x):
        return self.net(x)


class GIN(nn.Module):
    def __init__(self, n_out, hidden=128, layers=4, dropout=0.2):
        super().__init__()
        self.atom_emb = nn.ModuleList(nn.Embedding(len(v), hidden) for v in x_map.values())
        self.bond_emb = nn.ModuleList(nn.ModuleList(nn.Embedding(len(v), hidden) for v in e_map.values())
                                      for _ in range(layers))
        self.convs = nn.ModuleList(
            GINEConv(nn.Sequential(nn.Linear(hidden, 2 * hidden), nn.BatchNorm1d(2 * hidden), nn.ReLU(),
                                   nn.Linear(2 * hidden, hidden)), train_eps=True) for _ in range(layers))
        self.norms = nn.ModuleList(nn.BatchNorm1d(hidden) for _ in range(layers))
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, n_out))

    def forward(self, data):
        h = sum(emb(data.x[:, i]) for i, emb in enumerate(self.atom_emb))
        for conv, norm, bond_emb in zip(self.convs, self.norms, self.bond_emb):
            e = sum(emb(data.edge_attr[:, i]) for i, emb in enumerate(bond_emb))
            h = h + self.dropout(torch.relu(norm(conv(h, data.edge_index, e))))     # residual
        g = torch.cat([global_mean_pool(h, data.batch), global_max_pool(h, data.batch)], dim=1)
        return self.head(g)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def task_metrics(Y, P):
    """Per-assay [ROC-AUC, PR-AUC (average precision)] on observed labels -> array (12, 2)."""
    out = []
    for t in range(Y.shape[1]):
        m = ~np.isnan(Y[:, t])
        ok = len(np.unique(Y[m, t])) == 2
        out.append((roc_auc_score(Y[m, t], P[m, t]), average_precision_score(Y[m, t], P[m, t])) if ok
                   else (np.nan, np.nan))
    return np.array(out)


def masked_bce(logits, y, eps=0.0, T=None):
    """BCE over observed labels only. eps = label smoothing; T = (n_tasks, 2) noise channel [rho01, rho10]."""
    mask = ~torch.isnan(y)
    y0 = torch.nan_to_num(y)
    if T is not None:   # forward loss correction
        p = torch.sigmoid(logits)
        q = (p * (1 - T[:, 1]) + (1 - p) * T[:, 0]).clamp(1e-6, 1 - 1e-6)
        loss = -(y0 * torch.log(q) + (1 - y0) * torch.log(1 - q))
    else:
        loss = F.binary_cross_entropy_with_logits(logits, y0 * (1 - eps) + eps / 2, reduction="none")
    return (loss * mask).sum() / mask.sum().clamp(min=1)


class NNTrainer:
    """Trains the multi-task MLP (kind='mlp', on ECFP4) or GIN (kind='gnn', on graphs)."""

    def __init__(self, kind):
        self.kind = kind
        self.hp = dict(MLP_HP if kind == "mlp" else GNN_HP)
        if kind == "mlp":
            self.X = torch.tensor(X, dtype=torch.float32, device=DEVICE)

    def new_model(self):
        hp = self.hp
        m = MLP(ECFP_BITS, len(TASKS), hp["hidden"], hp["dropout"]) if self.kind == "mlp" else \
            GIN(len(TASKS), hp["hidden"], hp["layers"], hp["dropout"])
        return m.to(DEVICE)

    def batches(self, rows, shuffle, gen=None, batch_size=None):
        bs = batch_size or self.hp["batch_size"]
        if self.kind == "mlp":
            order = rows[torch.randperm(len(rows), generator=gen).numpy()] if shuffle else rows
            for i in range(0, len(order), bs):
                r = order[i:i + bs]
                if not (shuffle and len(r) < 2):          # BatchNorm needs >= 2 samples
                    yield self.X[r], torch.as_tensor(r)
        else:
            for b in DataLoader([GRAPHS[i] for i in rows], batch_size=bs, shuffle=shuffle, generator=gen):
                if not (shuffle and b.num_graphs < 2):
                    yield b.to(DEVICE), b.row.cpu()

    @torch.no_grad()
    def predict(self, model, rows):
        model.eval()
        return torch.cat([torch.sigmoid(model(x)).cpu() for x, _ in self.batches(rows, False, batch_size=512)]).numpy()

    def fit(self, Y, train_rows, val_rows=None, seed=0, eps=0.0, T=None, epochs=None):
        """Y = full label matrix used for training (and for validation, i.e. noisy too)."""
        set_seed(seed)
        hp = self.hp
        train_rows = train_rows[~np.isnan(Y[train_rows]).all(axis=1)]
        model = self.new_model()
        opt = torch.optim.Adam(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
        Yt = torch.tensor(Y, dtype=torch.float32)
        Tt = None if T is None else torch.tensor(T, dtype=torch.float32, device=DEVICE)
        gen = torch.Generator().manual_seed(seed)
        best, best_state, bad, n_epochs = -np.inf, None, 0, 0
        for epoch in range(epochs or hp["max_epochs"]):
            model.train()
            for x, r in self.batches(train_rows, True, gen):
                loss = masked_bce(model(x), Yt[r].to(DEVICE), eps, Tt)
                opt.zero_grad(); loss.backward(); opt.step()
            n_epochs = epoch + 1
            if val_rows is None:
                continue
            score = np.nanmean(task_metrics(Y[val_rows], self.predict(model, val_rows))[:, 0])
            if score > best:
                best, bad = score, 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= hp["patience"]:
                    break
        if best_state is not None:
            model.load_state_dict(best_state)
        model.best_val, model.n_epochs = best, n_epochs
        return model


def fit_rf(Y, rows, seed=0):
    """One random forest per assay on that assay's observed labels."""
    forests = []
    for t in range(Y.shape[1]):
        r = rows[~np.isnan(Y[rows, t])]
        forests.append(RandomForestClassifier(random_state=seed, **RF_HP).fit(X[r], Y[r, t].astype(int)))
    return forests


def predict_rf(forests, rows):
    return np.stack([f.predict_proba(X[rows])[:, 1] for f in forests], axis=1)


print("MLP parameters:", sum(p.numel() for p in MLP(ECFP_BITS, 12).parameters()),
      "| GIN parameters:", sum(p.numel() for p in GIN(12).parameters()))

## 5 · Noise detectors (used by the cleaning defences and by Task 4)
Both detectors start from **out-of-fold (OOF) predicted probabilities**. Every labelled molecule is scored by models trained without it (5-fold cross-validation), so a model cannot simply memorise a wrong label. Three deliberately different learners are used on ECFP4: L2 logistic regression, Tanimoto k-nearest-neighbours and a multi-task MLP.

* **Confident learning (CL)** (Northcutt et al. 2021, via its reference implementation `cleanlab`, `prune_by_noise_rate`), on the members' averaged OOF probabilities. CL also estimates each assay's noise transition matrix, which feeds the loss-correction defence.
* **Ensemble-disagreement filter (ED)**, a majority filter after Brodley & Friedl (1999). Each member votes "wrong label" when it confidently predicts the other class. "Confidently" uses class-specific thresholds: the member's mean predicted probability for that class, which is robust to the ~5 % active rate. A label with ≥ 2 of 3 votes is flagged.

A flagged label is set to **missing for that assay only**, so the molecule still trains the other 11 outputs.

In [ ]:
def tanimoto_knn(Xtr, ytr, Xq, k=10, alpha=1.0):
    """Similarity-weighted kNN P(active), shrunk towards the prior with `alpha` pseudo-counts."""
    A, B = Xq.astype(np.float32), Xtr.astype(np.float32)
    inter = A @ B.T
    sim = inter / (A.sum(1)[:, None] + B.sum(1)[None, :] - inter + 1e-9)
    nn_idx = np.argpartition(-sim, k - 1, axis=1)[:, :k]
    w = np.take_along_axis(sim, nn_idx, axis=1)
    return ((w * ytr[nn_idx]).sum(1) + alpha * ytr.mean()) / (w.sum(1) + alpha)


def oof_probabilities(Y, rows, seed=0):
    """{member: n x 12 matrix of out-of-fold P(active)} for the labelled entries of `rows`."""
    n, T = Y.shape
    out = {m: np.full((n, T), np.nan, dtype=np.float32) for m in DETECTOR_MEMBERS}
    mlp = NNTrainer("mlp")
    for k, (tr, va) in enumerate(KFold(OOF_FOLDS, shuffle=True, random_state=seed).split(rows)):
        tr_rows, va_rows = rows[tr], rows[va]
        for t in range(T):
            r = tr_rows[~np.isnan(Y[tr_rows, t])]
            out["lr"][va_rows, t] = LogisticRegression(C=0.1, max_iter=2000).fit(
                X[r], Y[r, t].astype(int)).predict_proba(X[va_rows])[:, 1]
            out["knn"][va_rows, t] = tanimoto_knn(X[r], Y[r, t], X[va_rows])
        model = mlp.fit(Y, tr_rows, seed=seed + k, epochs=DETECTOR_MLP_EPOCHS)
        out["mlp"][va_rows] = mlp.predict(model, va_rows)
    return out


def confident_learning(Y, rows, P):
    """Flags, label-quality score (low = suspect) and estimated noise channel (12 x [rho01, rho10])."""
    n, T = Y.shape
    flags, quality = np.zeros((n, T), dtype=bool), np.full((n, T), np.nan, dtype=np.float32)
    channel = np.zeros((T, 2), dtype=np.float32)
    for t in range(T):
        r = rows[~np.isnan(Y[rows, t])]
        y = Y[r, t].astype(int)
        p = np.clip(P[r, t].astype(np.float64), 1e-6, 1 - 1e-6)
        pp = np.stack([1 - p, p], axis=1)
        flags[r, t] = find_label_issues(y, pp, filter_by="prune_by_noise_rate")
        _, noise_matrix, _ = estimate_latent(compute_confident_joint(y, pp), y)  # [i, j] = P(label i | true j)
        quality[r, t] = pp[np.arange(len(y)), y]
        channel[t] = noise_matrix[1, 0], noise_matrix[0, 1]
    return flags, quality, channel


def ensemble_disagreement(Y, rows, P_members):
    """Flags (>= ED_MIN_VOTES members call the label wrong) and the vote fraction."""
    votes = np.zeros(Y.shape, dtype=np.float32)
    for P in P_members.values():
        for t in range(Y.shape[1]):
            r = rows[~np.isnan(Y[rows, t])]
            y, p = Y[r, t], P[r, t]
            t_pos, t_neg = p[y == 1].mean(), (1 - p[y == 0]).mean()   # class-specific thresholds
            votes[r, t] += np.where(y == 1, (1 - p) >= t_neg, p >= t_pos)
    frac = votes / len(P_members)
    frac[np.isnan(Y)] = np.nan
    return votes >= ED_MIN_VOTES, frac


def run_detectors(Y, rows, P_members):
    P_avg = np.mean(np.stack(list(P_members.values())), axis=0)
    cl_flags, cl_quality, channel = confident_learning(Y, rows, P_avg)
    ed_flags, ed_votes = ensemble_disagreement(Y, rows, P_members)
    return dict(cl_flags=cl_flags, cl_score=1 - cl_quality, channel=channel, ed_flags=ed_flags, ed_score=ed_votes)


def cached_oof(name, Y, rows, seed):
    """OOF probabilities are the slow part; cache them on Drive."""
    path = RESULTS_DIR / "cache" / f"oof_{name}_s{seed}.npz"
    if path.exists():
        z = np.load(path)
        return {m: z[m] for m in DETECTOR_MEMBERS}
    t0 = time.time()
    P = oof_probabilities(Y, rows, seed)
    np.savez_compressed(path, **P)
    print(f"  [{name} s{seed}] detector OOF probabilities computed in {time.time() - t0:.0f}s", flush=True)
    return P

## 6 · Tasks 2 + 3: the noise × defence × model × seed grid
For every noise condition (clean, `sym`/`cc` at 5/10/20 %) and seed, the grid does three things:
1. Injects the noise.
2. Computes the detector OOF probabilities once (cached), which give the CL/ED flags and the estimated noise channel.
3. Trains every (model, defence) pair on identical rows, seeds and hyperparameters, and scores it on the clean test set.

It also records how well each detector finds the injected flips (precision/recall).

Each run is saved to `results/runs/<model>_<defence>_<condition>_s<seed>.json` **as soon as it finishes**. Re-running a cell skips finished runs.

In [ ]:
RF_DEFENCES = {"none", "cl", "ed"}   # smoothing / loss correction do not apply to a random forest


def detection_report(det, flipped, rows):
    """Precision / recall of each detector against the labels we actually flipped."""
    obs = np.zeros_like(flipped)
    obs[rows] = ~np.isnan(ds.Y[rows])
    rep = {"n_labels": int(obs.sum()), "n_flipped": int(flipped.sum())}
    for name in ("cl", "ed"):
        f = det[f"{name}_flags"] & obs
        tp = int((f & flipped).sum())
        rep[name] = dict(flagged=int(f.sum()), precision=tp / max(int(f.sum()), 1),
                         recall=tp / int(flipped.sum()) if flipped.any() else float("nan"),
                         flag_rate_clean=float(f[obs & ~flipped].mean()))
    return rep


def run_grid(models, defences=DEFENCES, kinds=None, rates=None, seeds=None, include_clean=True, save_models=True):
    kinds, rates, seeds = kinds or NOISE_KINDS, rates or NOISE_RATES, seeds or SEEDS
    trainers = {m: NNTrainer(m) for m in models if m != "rf"}
    conditions = ([("none", 0.0)] if include_clean else []) + [(k, r) for k in kinds for r in rates]
    dev, te = ds.idx["dev"], ds.idx["test"]
    for d in ("runs", "detection"):
        (RESULTS_DIR / d).mkdir(exist_ok=True)
    for seed in seeds:
        for kind, rate in conditions:
            cond = condition_name(kind, rate)
            todo = [(m, d) for m in models for d in defences
                    if not (m == "rf" and d not in RF_DEFENCES) and not (d == "fc_oracle" and cond == "clean")
                    and not (RESULTS_DIR / "runs" / f"{m}_{d}_{cond}_s{seed}.json").exists()]
            if not todo:
                continue
            Yn, flipped = inject_noise(ds.Y, dev, kind, rate, seed)
            det = run_detectors(Yn, dev, cached_oof(cond, Yn, dev, seed))
            # oracle channel = the flip rates we actually injected (upper bound for loss correction)
            oracle = np.array([[flipped[dev, t][ds.Y[dev, t] == 0].mean(), flipped[dev, t][ds.Y[dev, t] == 1].mean()]
                               for t in range(len(TASKS))], dtype=np.float32)
            rep = detection_report(det, flipped, dev)
            rep.update(condition=cond, kind=kind, rate=rate, seed=seed,
                       channel_estimated=det["channel"].tolist(), channel_true=oracle.tolist())
            (RESULTS_DIR / "detection" / f"{cond}_s{seed}.json").write_text(json.dumps(rep))

            for model_name, defence in todo:
                t0 = time.time()
                Ytr, eps, T = Yn, 0.0, None                       # "none" = undefended baseline
                if defence == "ls":
                    eps = LABEL_SMOOTHING_EPS
                elif defence == "fc":
                    T = det["channel"]
                elif defence == "fc_oracle":
                    T = oracle
                elif defence in ("cl", "ed"):
                    Ytr = Yn.copy()
                    Ytr[det[f"{defence}_flags"]] = np.nan         # remove flagged labels (that assay only)
                info = {}
                if model_name == "rf":
                    P = predict_rf(fit_rf(Ytr, ds.idx["train"], seed), te)
                else:
                    tr = trainers[model_name]
                    model = tr.fit(Ytr, ds.idx["train"], ds.idx["valid"], seed=seed, eps=eps, T=T)
                    P = tr.predict(model, te)
                    info = dict(epochs=model.n_epochs, best_val_noisy_roc=float(model.best_val))
                    if save_models and seed == 0 and cond in SAVE_CONDITIONS:   # fp16 halves the file size
                        state = {k: v.half() if v.is_floating_point() else v for k, v in model.state_dict().items()}
                        torch.save({"state_dict": state, "hp": tr.hp, "kind": model_name, "tasks": TASKS},
                                   MODELS_DIR / "checkpoints" / f"{model_name}_{defence}_{cond}_s0.pt")
                m = task_metrics(ds.Y[te], P)
                rec = dict(model=model_name, defence=defence, condition=cond, kind=kind, rate=rate, seed=seed,
                           roc_auc=float(np.nanmean(m[:, 0])), pr_auc=float(np.nanmean(m[:, 1])),
                           roc_auc_tasks=dict(zip(TASKS, m[:, 0].tolist())), pr_auc_tasks=dict(zip(TASKS, m[:, 1].tolist())),
                           labels_removed=int(np.isnan(Ytr[dev]).sum() - np.isnan(Yn[dev]).sum()),
                           seconds=round(time.time() - t0, 1), **info)
                (RESULTS_DIR / "runs" / f"{model_name}_{defence}_{cond}_s{seed}.json").write_text(json.dumps(rec))
                print(f"  {model_name:3s} {defence:9s} {cond:9s} s{seed}  ROC {rec['roc_auc']:.3f}  PR {rec['pr_auc']:.3f}"
                      f"  removed {rec['labels_removed']:5d}  ({rec['seconds']:.0f}s)", flush=True)

### 6a · MLP + random forest (~1.5 h)
To split the work across the group, pass e.g. `seeds=[1]` on member B's account and `seeds=[2]` on member C's.

In [ ]:
run_grid(models=["mlp", "rf"])

### 6b · Graph neural network (~2–3 h, ~1 min per run)

In [ ]:
run_grid(models=["gnn"])

## 7 · Results: baseline vs. defences
Collects every run file and builds:
* **summary tables**: mean ± sd over seeds;
* **paired improvement**: each defence minus the undefended model of the same architecture, condition and seed;
* **`results/comparison_table.md`**: the baseline-vs-main slide table;
* **degradation curves**, a **gain heatmap**, and **detector precision/recall** against the injected flips.

In [ ]:
DEF_LABEL = {"none": "undefended (baseline)", "ls": "label smoothing", "fc": "loss correction (est. T)",
             "fc_oracle": "loss correction (oracle T)", "cl": "confident learning", "ed": "ensemble-disagreement filter"}
DEF_COLOR = {"none": "#222222", "ls": "#1f77b4", "fc": "#ff7f0e", "fc_oracle": "#ffbb78", "cl": "#2ca02c", "ed": "#d62728"}
MODEL_LABEL = {"rf": "RF (ECFP4)", "mlp": "MLP (ECFP4)", "gnn": "GIN (graph)"}
COND_ORDER = ["clean"] + [f"{k}_{r:.2f}" for k in NOISE_KINDS for r in NOISE_RATES]


def summarise_results():
    df = pd.DataFrame([json.loads(p.read_text()) for p in sorted((RESULTS_DIR / "runs").glob("*.json"))])
    df.drop(columns=["roc_auc_tasks", "pr_auc_tasks"]).to_csv(RESULTS_DIR / "grid.csv", index=False)
    s = df.groupby(["model", "defence", "condition", "kind", "rate"]).agg(
        roc_mean=("roc_auc", "mean"), roc_sd=("roc_auc", "std"), pr_mean=("pr_auc", "mean"), pr_sd=("pr_auc", "std"),
        n_seeds=("seed", "nunique"), labels_removed=("labels_removed", "mean")).reset_index()
    base = df[df.defence == "none"][["model", "condition", "seed", "roc_auc", "pr_auc"]]
    d = df.merge(base, on=["model", "condition", "seed"], suffixes=("", "_base"))
    d["d_roc"], d["d_pr"] = d.roc_auc - d.roc_auc_base, d.pr_auc - d.pr_auc_base
    delta = d[d.defence != "none"].groupby(["model", "defence", "condition"]).agg(
        d_roc_mean=("d_roc", "mean"), d_roc_sd=("d_roc", "std"), d_pr_mean=("d_pr", "mean"),
        d_pr_sd=("d_pr", "std"), wins=("d_roc", lambda x: f"{(x > 0).sum()}/{len(x)}")).reset_index()
    s.to_csv(RESULTS_DIR / "summary.csv", index=False)
    delta.to_csv(RESULTS_DIR / "delta_vs_undefended.csv", index=False)
    return s, delta


def comparison_table(s, delta, metric):
    conds = [c for c in COND_ORDER if c in set(s.condition)]
    lines = ["| Model | Defence | " + " | ".join(conds) + " |", "|" + "---|" * (len(conds) + 2)]
    for model in ("rf", "mlp", "gnn"):
        for dname in DEFENCES:
            sub = s[(s.model == model) & (s.defence == dname)].set_index("condition")
            if sub.empty:
                continue
            cells = []
            for c in conds:
                if c not in sub.index:
                    cells.append("–"); continue
                sd = sub.loc[c, f"{metric}_sd"]
                cell = f"{sub.loc[c, f'{metric}_mean']:.3f}" + (f" ± {sd:.3f}" if np.isfinite(sd) else "")
                dd = delta[(delta.model == model) & (delta.defence == dname) & (delta.condition == c)]
                if len(dd):
                    cell += f" ({dd[f'd_{metric}_mean'].iloc[0]:+.3f})"
                cells.append(cell)
            name = f"**{DEF_LABEL[dname]}**" if dname == "none" else DEF_LABEL[dname]
            lines.append(f"| {MODEL_LABEL[model]} | {name} | " + " | ".join(cells) + " |")
    return "\n".join(lines)


def plot_degradation(s, metric):
    models = [m for m in ("rf", "mlp", "gnn") if m in set(s.model)]
    fig, axes = plt.subplots(len(NOISE_KINDS), len(models), figsize=(4.2 * len(models), 3.5 * len(NOISE_KINDS)),
                             sharey="row", squeeze=False)
    for i, kind in enumerate(NOISE_KINDS):
        for j, model in enumerate(models):
            ax = axes[i, j]
            for dname in DEFENCES:
                sub = s[(s.model == model) & (s.defence == dname) & ((s.kind == kind) | (s.condition == "clean"))]
                if sub.empty:
                    continue
                sub = sub.sort_values("rate")
                ax.errorbar(sub.rate * 100, sub[f"{metric}_mean"], yerr=sub[f"{metric}_sd"].fillna(0), marker="o", ms=4,
                            capsize=2, lw=2.4 if dname == "none" else 1.3, ls="--" if dname == "fc_oracle" else "-",
                            color=DEF_COLOR[dname], label=DEF_LABEL[dname])
            ax.set_title(f"{MODEL_LABEL[model]} - {'symmetric' if kind == 'sym' else 'class-conditional'} noise", fontsize=9)
            ax.set_xlabel("injected noise rate (%)"); ax.grid(alpha=0.3)
            if j == 0:
                ax.set_ylabel(f"test {'ROC-AUC' if metric == 'roc' else 'PR-AUC'} (12-assay mean)")
    axes[0, -1].legend(fontsize=7, loc="lower left")
    fig.tight_layout(); fig.savefig(FIG_DIR / f"fig_degradation_{metric}.png", dpi=200); plt.show()


def plot_delta(delta):
    rows = [(m, d) for m in ("rf", "mlp", "gnn") for d in DEFENCES[1:] if len(delta[(delta.model == m) & (delta.defence == d)])]
    conds = [c for c in COND_ORDER if c in set(delta.condition)]
    M = np.full((len(rows), len(conds)), np.nan)
    for i, (m, d) in enumerate(rows):
        for j, c in enumerate(conds):
            v = delta[(delta.model == m) & (delta.defence == d) & (delta.condition == c)]
            if len(v):
                M[i, j] = v.d_roc_mean.iloc[0]
    lim = np.nanmax(np.abs(M))
    fig, ax = plt.subplots(figsize=(1.0 * len(conds) + 3, 0.4 * len(rows) + 1.5))
    im = ax.imshow(M, cmap="RdBu", vmin=-lim, vmax=lim, aspect="auto")
    ax.set_xticks(range(len(conds)), conds, rotation=30)
    ax.set_yticks(range(len(rows)), [f"{MODEL_LABEL[m]} | {DEF_LABEL[d]}" for m, d in rows], fontsize=7)
    for i in range(len(rows)):
        for j in range(len(conds)):
            if np.isfinite(M[i, j]):
                ax.text(j, i, f"{M[i, j]:+.3f}", ha="center", va="center", fontsize=6)
    fig.colorbar(im, ax=ax, label="Δ test ROC-AUC vs undefended (paired by seed)")
    ax.set_title("Defence gain over the undefended baseline")
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_delta_heatmap.png", dpi=200); plt.show()


def detection_summary():
    recs = []
    for p in sorted((RESULTS_DIR / "detection").glob("*.json")):
        r = json.loads(p.read_text())
        est, true = np.array(r["channel_estimated"]), np.array(r["channel_true"])
        for name in ("cl", "ed"):
            recs.append(dict(condition=r["condition"], kind=r["kind"], rate=r["rate"], seed=r["seed"], detector=name,
                             **{k: r[name][k] for k in ("flagged", "precision", "recall", "flag_rate_clean")},
                             rho01_est=est[:, 0].mean(), rho01_true=true[:, 0].mean(),
                             rho10_est=est[:, 1].mean(), rho10_true=true[:, 1].mean()))
    det = pd.DataFrame(recs)
    det.to_csv(RESULTS_DIR / "detection_summary.csv", index=False)
    g = det[det.kind != "none"].groupby(["kind", "rate", "detector"]).mean(numeric_only=True).reset_index()
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
    for kind, ls in (("sym", "-"), ("cc", "--")):
        for name in ("cl", "ed"):
            sub = g[(g.kind == kind) & (g.detector == name)]
            axes[0].plot(sub.rate * 100, sub.precision, ls=ls, marker="o", color=DEF_COLOR[name], label=f"{name.upper()} {kind}")
            axes[1].plot(sub.rate * 100, sub.recall, ls=ls, marker="o", color=DEF_COLOR[name], label=f"{name.upper()} {kind}")
        sub = g[(g.kind == kind) & (g.detector == "cl")]
        mk = "o" if kind == "sym" else "s"
        axes[2].scatter(sub.rho10_true, sub.rho10_est, marker=mk, color="#9467bd", label=f"ρ(1→0) {kind}")
        axes[2].scatter(sub.rho01_true, sub.rho01_est, marker=mk, color="#8c564b", label=f"ρ(0→1) {kind}")
    axes[0].set_title("Precision of flagged labels"); axes[1].set_title("Recall of injected flips")
    for ax in axes[:2]:
        ax.set_xlabel("injected noise rate (%)"); ax.set_ylim(0, 1); ax.grid(alpha=0.3)
    axes[0].legend(fontsize=7)
    axes[2].plot([0, 0.3], [0, 0.3], "k:", lw=1)
    axes[2].set_xlabel("true flip rate (mean over assays)"); axes[2].set_ylabel("CL-estimated flip rate")
    axes[2].set_title("Noise channel used by loss correction"); axes[2].legend(fontsize=7); axes[2].grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_detection.png", dpi=200); plt.show()
    return det.groupby(["condition", "detector"])[["flagged", "precision", "recall", "flag_rate_clean",
                                                   "rho01_est", "rho01_true", "rho10_est", "rho10_true"]].mean().round(3)


summary, delta = summarise_results()
det_table = detection_summary()
md = ["# Baseline vs. defended models\n",
      "Test metrics (clean labels, scaffold split), mean ± sd over seeds; in brackets: paired mean change vs. "
      "the undefended model of the same architecture, condition and seed.\n",
      "## ROC-AUC\n", comparison_table(summary, delta, "roc"), "\n## PR-AUC\n", comparison_table(summary, delta, "pr"),
      "\n## Noise detection vs. injected flips\n", det_table.reset_index().to_markdown(index=False)]
(RESULTS_DIR / "comparison_table.md").write_text("\n".join(md) + "\n")
display(Markdown("\n".join(md)))
plot_degradation(summary, "roc")
plot_degradation(summary, "pr")
plot_delta(delta)

## 8 · Task 4: do the detectors flag *real* suspect labels?
The detectors are run on the **natural** (un-noised) labels of all 7.5k molecules. A *suspect* label is one where the two related-assay readouts disagree; a *concordant* label is one where they agree.

Because the probabilities are out-of-fold, a molecule's partner-assay label is never seen when its own label is scored. Any enrichment is therefore not leakage: it means the label does not fit the structure–activity pattern learned from *other* molecules.

Suspects are strongly enriched in actives, and detectors flag actives more often. The fair comparison is therefore **within label class** (suspect actives vs. concordant actives, suspect inactives vs. concordant inactives). The statistics reported are:
* one-sided Fisher exact test;
* lift and precision/recall;
* AUC of the continuous detector score.

In [ ]:
def compare_flags(flags, score, suspect, population):
    s, c = population & suspect, population & ~suspect
    a, b = int((flags & s).sum()), int((~flags & s).sum())      # suspects: flagged / not flagged
    cc, d = int((flags & c).sum()), int((~flags & c).sum())     # concordant: flagged / not flagged
    odds, p = fisher_exact([[a, b], [cc, d]], alternative="greater")
    y = np.r_[np.ones(s.sum()), np.zeros(c.sum())]
    return dict(n_suspect=a + b, n_concordant=cc + d, flag_rate_suspect=a / max(a + b, 1),
                flag_rate_concordant=cc / max(cc + d, 1), lift=(a / max(a + b, 1)) / max(cc / max(cc + d, 1), 1e-9),
                odds_ratio=odds, fisher_p=p, precision=a / max(a + cc, 1), recall=a / max(a + b, 1),
                score_auc=roc_auc_score(y, np.r_[score[s], score[c]]))


def validate_suspects(seeds=None):
    seeds = seeds or SEEDS
    Y, rows = ds.Y, np.arange(len(ds.Y))
    suspect, paired = suspect_matrix(Y)
    Yc = np.nan_to_num(Y, nan=-1)
    strata = {"all paired labels": paired, "actives only": paired & (Yc == 1), "inactives only": paired & (Yc == 0)}
    for a, b in RELATED_PAIRS:
        for t in (a, b):
            m = np.zeros_like(paired)
            m[:, TASKS.index(t)] = True
            strata[t] = paired & m
    recs, dirs = [], []
    for seed in seeds:
        det = run_detectors(Y, rows, cached_oof("natural_all", Y, rows, seed))
        for key in ("cl", "ed"):
            flags, score = det[f"{key}_flags"], np.nan_to_num(det[f"{key}_score"])
            for sname, pop in strata.items():
                recs.append(dict(seed=seed, detector=key, stratum=sname, **compare_flags(flags, score, suspect, pop)))
            for a, b in RELATED_PAIRS:   # which of the two disagreeing labels gets flagged?
                i, j = TASKS.index(a), TASKS.index(b)
                for act, inact in ((i, j), (j, i)):
                    m = (Yc[:, act] == 1) & (Yc[:, inact] == 0)
                    dirs.append(dict(seed=seed, detector=key, active_in=TASKS[act], inactive_in=TASKS[inact], n=int(m.sum()),
                                     active_label_flagged=flags[m, act].mean(), inactive_label_flagged=flags[m, inact].mean()))
    res = pd.DataFrame(recs)
    res.to_csv(RESULTS_DIR / "suspects_per_seed.csv", index=False)
    summ = res.groupby(["detector", "stratum"], sort=False).agg(
        n_suspect=("n_suspect", "first"), n_concordant=("n_concordant", "first"),
        flag_rate_suspect=("flag_rate_suspect", "mean"), flag_rate_concordant=("flag_rate_concordant", "mean"),
        lift=("lift", "mean"), odds_ratio=("odds_ratio", "mean"), fisher_p_max=("fisher_p", "max"),
        precision=("precision", "mean"), recall=("recall", "mean"), score_auc=("score_auc", "mean")).reset_index()
    summ.to_csv(RESULTS_DIR / "suspects_summary.csv", index=False)
    dr = pd.DataFrame(dirs).groupby(["detector", "active_in", "inactive_in"], sort=False).mean(numeric_only=True)
    dr = dr.drop(columns="seed").reset_index()
    dr.to_csv(RESULTS_DIR / "suspects_direction.csv", index=False)
    md = ["# Task 4 - detector flags vs. real suspect labels\n", f"Seeds: {seeds}. `fisher_p_max` = largest one-sided "
          "Fisher p over seeds; `score_auc` = AUC of the detector score for suspect vs concordant (0.5 = no signal).\n",
          summ.round(4).assign(fisher_p_max=summ.fisher_p_max.map(lambda v: f"{v:.1e}")).to_markdown(index=False),
          "\n## Which label of a disagreeing pair is flagged?\n", dr.round(3).to_markdown(index=False)]
    (RESULTS_DIR / "suspects_report.md").write_text("\n".join(md) + "\n")
    display(Markdown("\n".join(md)))

    per = res[res.stratum.isin(["all paired labels", "actives only", "inactives only"])]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
    for ax, key, name in ((axes[0], "cl", "confident learning"), (axes[1], "ed", "ensemble disagreement")):
        sub = per[per.detector == key].groupby("stratum", sort=False)
        m, sd, pmax = sub.mean(numeric_only=True), sub.std(numeric_only=True).fillna(0), sub.fisher_p.max()
        x = np.arange(len(m))
        ax.bar(x - 0.2, m.flag_rate_suspect * 100, 0.4, yerr=sd.flag_rate_suspect * 100, capsize=3, color="#d62728",
               label="suspect (related assays disagree)")
        ax.bar(x + 0.2, m.flag_rate_concordant * 100, 0.4, yerr=sd.flag_rate_concordant * 100, capsize=3, color="#7f7f7f",
               label="concordant (related assays agree)")
        for k in range(len(m)):
            p = pmax.iloc[k]
            ax.text(k, max(m.flag_rate_suspect.iloc[k], m.flag_rate_concordant.iloc[k]) * 100 + 2,
                    f"lift {m.lift.iloc[k]:.1f}x\n" + (f"p={p:.0e}" if p > 0 else "p<1e-300"), ha="center", fontsize=7)
        ax.set_xticks(x, m.index); ax.set_title(name); ax.set_ylabel("labels flagged as noisy (%)"); ax.grid(alpha=0.3, axis="y")
    axes[0].set_ylim(0, per.groupby(["detector", "stratum"]).flag_rate_suspect.mean().max() * 130)
    axes[0].legend(fontsize=7, loc="upper right")
    fig.suptitle("Do detectors flag real suspect labels more often than concordant ones?", fontsize=10)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_suspects.png", dpi=200); plt.show()


validate_suspects()

## 9 · Stretch: multi-task noise interaction
Does label noise in some assays hurt the *other*, clean assays through the shared network? Does cleaning the noisy assays help them?

Heavy symmetric noise (20 %) is injected into the 7 nuclear-receptor (NR) assays only; the 5 stress-response (SR) assays stay clean. Four training regimes:

| Regime | What is trained |
|---|---|
| `clean` | No synthetic noise (reference) |
| `noisy` | NR assays noisy, no defence |
| `noisy+CL` | Confident learning cleans the NR assays only |
| `drop_noisy` | NR labels discarded: a 5-assay model on the SR assays only |

In [ ]:
NOISY_TASKS = [t for t in TASKS if t.startswith("NR-")]
CLEAN_TASKS = [t for t in TASKS if t not in NOISY_TASKS]
REGIMES = ["clean", "noisy", "noisy+CL", "drop_noisy"]


def noise_interaction(models=("mlp", "gnn"), kind="sym", rate=0.20, seeds=None):
    seeds = seeds or SEEDS
    nz, cl_idx = [TASKS.index(t) for t in NOISY_TASKS], [TASKS.index(t) for t in CLEAN_TASKS]
    dev, te = ds.idx["dev"], ds.idx["test"]
    out_dir = RESULTS_DIR / "interaction"
    out_dir.mkdir(exist_ok=True)
    for seed in seeds:
        Yall, _ = inject_noise(ds.Y, dev, kind, rate, seed)
        Yn = ds.Y.copy()
        Yn[:, nz] = Yall[:, nz]                                    # noise only in the NR assays
        det = run_detectors(Yn, dev, cached_oof(f"interaction_{kind}_{rate:.2f}", Yn, dev, seed))
        flags = det["cl_flags"].copy()
        flags[:, cl_idx] = False                                   # clean only the noisy assays
        labels = {"clean": ds.Y, "noisy": Yn, "noisy+CL": np.where(flags, np.nan, Yn), "drop_noisy": ds.Y.copy()}
        labels["drop_noisy"][:, nz] = np.nan
        for model_name in models:
            tr = NNTrainer(model_name)
            for regime in REGIMES:
                path = out_dir / f"{model_name}_{regime}_{kind}_{rate:.2f}_s{seed}.json"
                if path.exists():
                    continue
                model = tr.fit(labels[regime], ds.idx["train"], ds.idx["valid"], seed=seed)
                m = task_metrics(ds.Y[te], tr.predict(model, te))
                rec = dict(model=model_name, regime=regime, seed=seed,
                           roc_clean_SR=float(np.nanmean(m[cl_idx, 0])), pr_clean_SR=float(np.nanmean(m[cl_idx, 1])),
                           roc_noisy_NR=None if regime == "drop_noisy" else float(np.nanmean(m[nz, 0])),
                           pr_noisy_NR=None if regime == "drop_noisy" else float(np.nanmean(m[nz, 1])))
                path.write_text(json.dumps(rec))
                print(f"  {model_name} {regime:10s} s{seed}  clean-SR ROC {rec['roc_clean_SR']:.3f}", flush=True)
    df = pd.DataFrame([json.loads(p.read_text()) for p in sorted(out_dir.glob("*.json"))])
    df.to_csv(RESULTS_DIR / "interaction.csv", index=False)
    summ = df.groupby(["model", "regime"])[["roc_clean_SR", "pr_clean_SR", "roc_noisy_NR", "pr_noisy_NR"]].agg(["mean", "std"])
    summ.columns = ["_".join(c) for c in summ.columns]
    md = ["# Stretch - multi-task noise interaction\n", f"Noisy (NR): {', '.join(NOISY_TASKS)}. Clean (SR): "
          f"{', '.join(CLEAN_TASKS)}. Noise: {kind} {rate:.0%}.\n", summ.round(4).reset_index().to_markdown(index=False)]
    (RESULTS_DIR / "interaction_report.md").write_text("\n".join(md) + "\n")
    display(Markdown("\n".join(md)))

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    for ax, col, title in ((axes[0], "roc_clean_SR", "clean SR assays"), (axes[1], "roc_noisy_NR", "noisy NR assays")):
        ms = list(dict.fromkeys(df.model))
        for k, model_name in enumerate(ms):
            g = df[df.model == model_name].groupby("regime")[col]
            ax.bar(np.arange(4) + (k - 0.5 * (len(ms) - 1)) * 0.35, g.mean().reindex(REGIMES), 0.35,
                   yerr=g.std().reindex(REGIMES).fillna(0), capsize=3, label=model_name.upper())
        ax.set_xticks(range(4), REGIMES); ax.set_ylim(0.5, None); ax.set_title(f"test ROC-AUC on {title}")
        ax.grid(alpha=0.3, axis="y")
    axes[0].legend()
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_interaction.png", dpi=200); plt.show()


noise_interaction()

## 10 · Model cards, package versions and the submission zip
Writes one model card per (model, defence) to `models/cards/`. Each card is filled with the architecture, hyperparameters, data version and test results. The cell also records the package versions and zips the deliverables:
* curated dataset and curation log;
* results and figures;
* model cards and seed-0 checkpoints.

Add this notebook (*File → Download → .ipynb*) and your slides to the submission.

In [ ]:
ARCH = {"rf": f"Random forest per assay on {ECFP_BITS}-bit ECFP4. {RF_HP}.",
        "mlp": f"Multi-task MLP on {ECFP_BITS}-bit ECFP4, hidden {MLP_HP['hidden']}, BatchNorm/ReLU/dropout "
               f"{MLP_HP['dropout']}, 12 sigmoid outputs, masked BCE, Adam lr {MLP_HP['lr']}, early stopping "
               f"(patience {MLP_HP['patience']}) on validation ROC-AUC. ~2.4 M parameters.",
        "gnn": f"Multi-task GINE network (PyTorch Geometric), {GNN_HP['layers']} residual layers of width "
               f"{GNN_HP['hidden']}, mean+max pooling, 12 sigmoid outputs, masked BCE, Adam lr {GNN_HP['lr']}, "
               f"early stopping (patience {GNN_HP['patience']}). ~0.34 M parameters."}
DEF_TEXT = {"none": "Standard training on the (possibly noisy) labels - the baseline.",
            "ls": f"Label smoothing, eps = {LABEL_SMOOTHING_EPS}.",
            "fc": "Forward loss correction with the noise channel estimated by confident learning.",
            "fc_oracle": "Forward loss correction with the true injected flip rates (upper bound, not deployable).",
            "cl": "Confident learning (cleanlab prune_by_noise_rate) on out-of-fold probabilities of LR + kNN + MLP; "
                  "flagged labels set to missing.",
            "ed": f"Ensemble-disagreement filter: >= {ED_MIN_VOTES} of LR / kNN / MLP confidently disagree with the "
                  "label -> label set to missing."}


def write_model_cards():
    sha = hashlib.sha256(CURATED_PATH.read_bytes()).hexdigest()[:16]
    for model_name, defence in [("rf", d) for d in ("none", "cl", "ed")] + [(m, d) for m in ("mlp", "gnn") for d in DEFENCES]:
        sub = summary[(summary.model == model_name) & (summary.defence == defence)].set_index("condition")
        lines = [f"# Model card - {MODEL_LABEL[model_name]}, {DEF_LABEL[defence]}\n",
                 f"**Role:** {'baseline' if defence == 'none' else 'main model (label-noise defence)'}\n",
                 f"## Architecture\n{ARCH[model_name]}\n", f"## Noise handling\n{DEF_TEXT[defence]}\n",
                 f"## Training data\n`data/tox21_curated.csv` (sha256 `{sha}`...), {len(ds.Y)} compounds, 12 Tox21 assays, "
                 "scaffold split 80/10/10. Noise conditions corrupt train+valid labels only; test labels are original.\n",
                 f"## Checkpoint\n`models/checkpoints/{model_name}_{defence}_<condition>_s0.pt`, seed 0, conditions "
                 f"{SAVE_CONDITIONS}. Weights stored as float16: load with `model.load_state_dict({{k: v.float() for k, v "
                 "in ckpt['state_dict'].items()}})`.\n" if model_name != "rf" else "## Checkpoint\nRF is retrained in seconds; not saved.\n",
                 "## Test performance (mean ± sd over seeds)\n| condition | ROC-AUC | PR-AUC | labels removed |\n|---|---|---|---|"]
        for c in COND_ORDER:
            if c in sub.index:
                r = sub.loc[c]
                lines.append(f"| {c} | {r.roc_mean:.3f} ± {r.roc_sd:.3f} | {r.pr_mean:.3f} ± {r.pr_sd:.3f} | {r.labels_removed:.0f} |")
        lines += ["\n## Intended use\nTeaching/research on label noise in in-vitro Tox21-type assay prediction.\n",
                  "## Limitations\n* Not a regulatory/safety tool; in-vitro HTS readouts, not in-vivo toxicity.\n"
                  "* Small per-assay test sets (20-117 actives): use the 12-assay mean and seed spread.\n"
                  "* Parent-fragment standardisation ignores counter-ion/mixture effects.\n"
                  "* Synthetic noise is structure-independent, unlike real assay error."]
        (MODELS_DIR / "cards" / f"{model_name}_{defence}.md").write_text("\n".join(lines) + "\n")


import cleanlab, rdkit, sklearn, torch_geometric
versions = {"python": platform.python_version(), "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0)
            if torch.cuda.is_available() else None, "torch_geometric": torch_geometric.__version__,
            "rdkit": rdkit.__version__, "cleanlab": cleanlab.__version__, "scikit-learn": sklearn.__version__,
            "numpy": np.__version__, "pandas": pd.__version__}
(RESULTS_DIR / "versions.json").write_text(json.dumps(versions, indent=1))
print(versions)
write_model_cards()

import zipfile
zip_path = ("/content/" if IN_COLAB else "../") + "CM4044_MP2-30_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:      # everything except the caches
    for top in ("data", RESULTS_DIR.name, FIG_DIR.name, "models"):
        for p in sorted(Path(top).rglob("*")):
            if p.is_file() and "cache" not in p.parts:
                z.write(p)
print(f"{zip_path}: {os.path.getsize(zip_path) / 1e6:.0f} MB")
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)